# Assignment 2: Mini Project 2 - Software Development with Gen AI
This notebook builds a simulated deployment of **CarePulse (Hospital Appointment Booking System)**. It covers:
- **Phase 4 (Development)**: SQLite database setup, core querying, business logic (booking confirmation + billing invoice), and a **highly aesthetic, interactive HTML frontend dashboard**.
- **Phase 5 (Testing)**: Automated structural, quality, business, and edge case double-booking tests.
- **Phase 6 (Deployment & Monitoring)**: A 10-minute response-time monitoring simulation with a traffic spike and auto-remedy.
- **Phase 7 (Maintenance & Prediction)**: 30-day capacity tracking and threshold forecasting using linear regression.

## Phase 4 — Development
We set up our SQLite database `carepulse.db`, insert mock records, declare searching & booking logic, and render a premium-styled dashboard using `IPython.display.HTML`.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

db_path = 'carepulse.db'
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON;')

# 1. DATABASE SETUP (Module 1)
cursor.executescript('''
CREATE TABLE patients (
    patient_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    phone TEXT NOT NULL,
    dob TEXT NOT NULL
);

CREATE TABLE doctors (
    doctor_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    specialty TEXT NOT NULL,
    experience_years INTEGER NOT NULL,
    consultation_fee REAL NOT NULL
);

CREATE TABLE appointments (
    appointment_id INTEGER PRIMARY KEY AUTOINCREMENT,
    patient_id INTEGER NOT NULL,
    doctor_id INTEGER NOT NULL,
    appointment_date TEXT NOT NULL,
    time_slot TEXT NOT NULL,
    status TEXT DEFAULT 'Pending',
    FOREIGN KEY(patient_id) REFERENCES patients(patient_id),
    FOREIGN KEY(doctor_id) REFERENCES doctors(doctor_id)
);

CREATE TABLE billing (
    billing_id INTEGER PRIMARY KEY AUTOINCREMENT,
    appointment_id INTEGER NOT NULL,
    amount_due REAL NOT NULL CHECK(amount_due >= 0),
    payment_status TEXT DEFAULT 'Unpaid',
    FOREIGN KEY(appointment_id) REFERENCES appointments(appointment_id)
);
''')
conn.commit()

# Insert Doctors
doctors_data = [
    ("Dr. Alice Smith", "Cardiology", 15, 150.0),
    ("Dr. Bob Jones", "Pediatrics", 10, 100.0),
    ("Dr. Carol Vance", "Orthopedics", 12, 120.0),
    ("Dr. David Miller", "Dermatology", 8, 90.0),
    ("Dr. Emma Watson", "Neurology", 14, 180.0)
]
cursor.executemany('INSERT INTO doctors (name, specialty, experience_years, consultation_fee) VALUES (?, ?, ?, ?);', doctors_data)

# Insert Patients
patients_data = [
    ("John Doe", "john.doe@email.com", "555-0192", "1990-05-15"),
    ("Jane Smith", "jane.smith@email.com", "555-0143", "1985-08-22"),
    ("Alex Johnson", "alex.j@email.com", "555-0177", "1998-12-10")
]
cursor.executemany('INSERT INTO patients (name, email, phone, dob) VALUES (?, ?, ?, ?);', patients_data)
conn.commit()

print('--- Verification Counts ---')
for t in ['patients', 'doctors']:
    cursor.execute(f'SELECT COUNT(*) FROM {t};')
    print(f"{t.capitalize()} count: {cursor.fetchone()[0]}")


# 2. CORE DATA FUNCTIONS (Module 2)
def search_doctors(specialty=None, max_fee=None):
    query = "SELECT * FROM doctors WHERE 1=1"
    params = []
    if specialty:
        query += " AND specialty = ?"
        params.append(specialty)
    if max_fee:
        query += " AND consultation_fee <= ?"
        params.append(max_fee)
    
    c = conn.cursor()
    c.execute(query, params)
    return c.fetchall()


# 3. MAIN BUSINESS FUNCTION (Module 3)
def book_appointment(patient_id, doctor_id, date, slot):
    c = conn.cursor()
    # Validate doctor exists
    c.execute("SELECT consultation_fee FROM doctors WHERE doctor_id = ?", (doctor_id,))
    doc = c.fetchone()
    if not doc:
        raise ValueError(f"Doctor ID {doctor_id} does not exist.")
    fee = doc[0]
    
    # Double-booking check
    c.execute('''
        SELECT COUNT(*) FROM appointments 
        WHERE doctor_id = ? AND appointment_date = ? AND time_slot = ? AND status != 'Cancelled'
    ''', (doctor_id, date, slot))
    if c.fetchone()[0] > 0:
        raise ValueError("Doctor is already booked for this date and time slot.")
        
    # Book appointment
    c.execute('INSERT INTO appointments (patient_id, doctor_id, appointment_date, time_slot, status) VALUES (?, ?, ?, ?, "Confirmed");', 
              (patient_id, doctor_id, date, slot))
    app_id = c.lastrowid
    
    # Create bill
    c.execute('INSERT INTO billing (appointment_id, amount_due, payment_status) VALUES (?, ?, "Unpaid");', (app_id, fee))
    conn.commit()
    return app_id

# Seed some appointments
book_appointment(1, 1, '2026-06-25', '09:00 AM')
book_appointment(2, 2, '2026-06-25', '11:00 AM')
book_appointment(1, 3, '2026-06-26', '02:00 PM')

### Module 4: HTML Frontend
We construct and render a premium doctor catalog and booking list using modern, responsive styling directly within Colab.

In [ ]:
from IPython.display import HTML

# Query data for frontend
cursor.execute('SELECT * FROM doctors;')
docs = cursor.fetchall()

cursor.execute('''
    SELECT a.appointment_id, p.name, d.name, d.specialty, a.appointment_date, a.time_slot, a.status, b.amount_due, b.payment_status
    FROM appointments a
    JOIN patients p ON a.patient_id = p.patient_id
    JOIN doctors d ON a.doctor_id = d.doctor_id
    JOIN billing b ON a.appointment_id = b.appointment_id;
''')
apps = cursor.fetchall()

# Generate HTML doctor list
doc_cards_html = ""
for d in docs:
    doc_cards_html += f"""
    <div class="doctor-card">
        <div class="doc-avatar">{d[1][4:6]}</div>
        <h3>{d[1]}</h3>
        <span class="doc-specialty">{d[2]}</span>
        <p class="doc-meta"><strong>Experience:</strong> {d[3]} Years</p>
        <p class="doc-meta"><strong>Consultation Fee:</strong> ${d[4]:.2f}</p>
        <button class="btn btn-book">Book Slot</button>
    </div>
    """

# Generate HTML appointments list
app_rows_html = ""
for a in apps:
    status_class = "status-confirmed" if a[6] == 'Confirmed' else "status-pending"
    bill_class = "bill-paid" if a[8] == 'Paid' else "bill-unpaid"
    app_rows_html += f"""
    <tr>
        <td>#{a[0]}</td>
        <td>{a[1]}</td>
        <td><strong>{a[2]}</strong><br><small style='color:#718096'>{a[3]}</small></td>
        <td>{a[4]}<br><small>{a[5]}</small></td>
        <td><span class="badge {status_class}">{a[6]}</span></td>
        <td>${a[7]:.2f}</td>
        <td><span class="badge {bill_class}">{a[8]}</span></td>
    </tr>
    """

# Master HTML template
portal_html = f"""
<link href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>
    .portal-body {{
        font-family: 'Plus Jakarta Sans', sans-serif;
        background-color: #f7fafc;
        color: #2d3748;
        padding: 24px;
        border-radius: 12px;
        box-shadow: 0 10px 15px -3px rgba(0, 0, 0, 0.05);
        max-width: 1100px;
        margin: 0 auto;
    }}
    .portal-header {{
        display: flex;
        justify-content: space-between;
        align-items: center;
        padding-bottom: 20px;
        border-bottom: 2px solid #edf2f7;
        margin-bottom: 30px;
    }}
    .portal-logo {{
        font-size: 24px;
        font-weight: 700;
        color: #4c51bf;
        display: flex;
        align-items: center;
        gap: 8px;
    }}
    .portal-logo span {{
        color: #319795;
    }}
    .hero-section {{
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        border-radius: 12px;
        padding: 30px;
        margin-bottom: 35px;
        box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);
    }}
    .hero-section h1 {{
        margin: 0 0 10px 0;
        font-size: 28px;
        font-weight: 700;
    }}
    .hero-section p {{
        margin: 0;
        font-size: 16px;
        opacity: 0.9;
    }}
    .section-title {{
        font-size: 20px;
        font-weight: 600;
        margin-bottom: 20px;
        color: #1a202c;
        border-left: 4px solid #4c51bf;
        padding-left: 10px;
    }}
    .doctor-grid {{
        display: grid;
        grid-template-columns: repeat(auto-fill, minmax(200px, 1fr));
        gap: 20px;
        margin-bottom: 40px;
    }}
    .doctor-card {{
        background: white;
        border-radius: 12px;
        padding: 20px;
        text-align: center;
        box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
        border: 1px solid #edf2f7;
        transition: all 0.3s ease;
    }}
    .doctor-card:hover {{
        transform: translateY(-5px);
        box-shadow: 0 10px 15px -3px rgba(76, 81, 191, 0.1);
        border-color: #4c51bf;
    }}
    .doc-avatar {{
        width: 50px;
        height: 50px;
        border-radius: 50%;
        background: #e0e7ff;
        color: #4c51bf;
        margin: 0 auto 12px auto;
        display: flex;
        align-items: center;
        justify-content: center;
        font-weight: 700;
        font-size: 18px;
    }}
    .doctor-card h3 {{
        margin: 0 0 4px 0;
        font-size: 16px;
        font-weight: 600;
    }}
    .doc-specialty {{
        display: inline-block;
        background: #e6fffa;
        color: #319795;
        font-size: 12px;
        font-weight: 600;
        padding: 2px 8px;
        border-radius: 99px;
        margin-bottom: 12px;
    }}
    .doc-meta {{
        font-size: 13px;
        color: #4a5568;
        margin: 4px 0;
    }}
    .btn {{
        background: #4c51bf;
        color: white;
        border: none;
        padding: 8px 16px;
        border-radius: 6px;
        font-size: 13px;
        font-weight: 600;
        cursor: pointer;
        transition: background 0.2s;
        width: 100%;
        margin-top: 10px;
    }}
    .btn:hover {{
        background: #3c366b;
    }}
    .table-container {{
        background: white;
        border-radius: 12px;
        box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
        border: 1px solid #edf2f7;
        overflow: hidden;
        margin-bottom: 20px;
    }}
    table {{
        width: 100%;
        border-collapse: collapse;
        text-align: left;
    }}
    th, td {{
        padding: 14px 20px;
        font-size: 14px;
        border-bottom: 1px solid #edf2f7;
    }}
    th {{
        background: #f7fafc;
        color: #4a5568;
        font-weight: 600;
    }}
    .badge {{
        display: inline-block;
        padding: 4px 8px;
        border-radius: 99px;
        font-size: 12px;
        font-weight: 600;
    }}
    .status-confirmed {{
        background: #e6fffa;
        color: #234e52;
    }}
    .status-pending {{
        background: #fffaf0;
        color: #7b341e;
    }}
    .bill-paid {{
        background: #ebf8ff;
        color: #2b6cb0;
    }}
    .bill-unpaid {{
        background: #fff5f5;
        color: #9b2c2c;
    }}
</style>
<div class="portal-body">
    <div class="portal-header">
        <div class="portal-logo">Care<span>Pulse</span></div>
        <div style="font-size: 14px; font-weight: 500; color: #4a5568;">Patient Portal</div>
    </div>
    <div class="hero-section">
        <h1>Welcome back, John!</h1>
        <p>Book new consultations or view your appointment billing invoices below.</p>
    </div>
    
    <h2 class="section-title">Available Specialists</h2>
    <div class="doctor-grid">
        {doc_cards_html}
    </div>
    
    <h2 class="section-title">Active Booking Tracker</h2>
    <div class="table-container">
        <table>
            <thead>
                <tr>
                    <th>Booking ID</th>
                    <th>Patient</th>
                    <th>Doctor</th>
                    <th>Schedule</th>
                    <th>Status</th>
                    <th>Consultation Fee</th>
                    <th>Billing Invoice</th>
                </tr>
            </thead>
            <tbody>
                {app_rows_html}
            </tbody>
        </table>
    </div>
</div>
"""
HTML(portal_html)

## Phase 5 — Testing
We build an automated test suite checking database structure, data quality constraints, business logic, and slot conflict/double-booking prevention.

In [ ]:
import unittest

class TestCarePulse(unittest.TestCase):
    def setUp(self):
        self.conn = sqlite3.connect(':memory:')
        self.cursor = self.conn.cursor()
        self.cursor.execute('PRAGMA foreign_keys = ON;')
        
        # Schema Setup
        self.cursor.executescript('''
        CREATE TABLE patients (
            patient_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            email TEXT UNIQUE NOT NULL,
            phone TEXT NOT NULL,
            dob TEXT NOT NULL
        );
        CREATE TABLE doctors (
            doctor_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            specialty TEXT NOT NULL,
            experience_years INTEGER NOT NULL,
            consultation_fee REAL NOT NULL
        );
        CREATE TABLE appointments (
            appointment_id INTEGER PRIMARY KEY AUTOINCREMENT,
            patient_id INTEGER NOT NULL,
            doctor_id INTEGER NOT NULL,
            appointment_date TEXT NOT NULL,
            time_slot TEXT NOT NULL,
            status TEXT DEFAULT 'Pending',
            FOREIGN KEY(patient_id) REFERENCES patients(patient_id),
            FOREIGN KEY(doctor_id) REFERENCES doctors(doctor_id)
        );
        CREATE TABLE billing (
            billing_id INTEGER PRIMARY KEY AUTOINCREMENT,
            appointment_id INTEGER NOT NULL,
            amount_due REAL NOT NULL CHECK(amount_due >= 0),
            payment_status TEXT DEFAULT 'Unpaid',
            FOREIGN KEY(appointment_id) REFERENCES appointments(appointment_id)
        );
        ''')
        
        # Seed
        self.cursor.execute('INSERT INTO doctors (name, specialty, experience_years, consultation_fee) VALUES ("Dr. Smith", "Cardiology", 15, 150.0);')
        self.cursor.execute('INSERT INTO patients (name, email, phone, dob) VALUES ("John Doe", "john@email.com", "555-1234", "1990-01-01");')
        self.conn.commit()

    def test_database_structure(self):
        self.cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [r[0] for r in self.cursor.fetchall()]
        self.assertTrue('patients' in tables)
        self.assertTrue('doctors' in tables)
        self.assertTrue('appointments' in tables)
        self.assertTrue('billing' in tables)

    def test_data_quality_constraints(self):
        # Check check-constraint on billing amount_due
        with self.assertRaises(sqlite3.IntegrityError):
            self.cursor.execute('INSERT INTO appointments (patient_id, doctor_id, appointment_date, time_slot) VALUES (1, 1, "2026-06-25", "10:00 AM");')
            app_id = self.cursor.lastrowid
            self.cursor.execute('INSERT INTO billing (appointment_id, amount_due) VALUES (?, -50.0);', (app_id,))
            self.conn.commit()

    def test_booking_and_billing(self):
        # Helper book function inside test
        def local_book(pid, did, date, slot):
            self.cursor.execute('SELECT consultation_fee FROM doctors WHERE doctor_id = ?', (did,))
            fee = self.cursor.fetchone()[0]
            self.cursor.execute('INSERT INTO appointments (patient_id, doctor_id, appointment_date, time_slot, status) VALUES (?, ?, ?, ?, "Confirmed");', (pid, did, date, slot))
            aid = self.cursor.lastrowid
            self.cursor.execute('INSERT INTO billing (appointment_id, amount_due, payment_status) VALUES (?, ?, "Unpaid");', (aid, fee))
            self.conn.commit()
            return aid
            
        app_id = local_book(1, 1, '2026-06-25', '10:00 AM')
        self.cursor.execute('SELECT status FROM appointments WHERE appointment_id = ?', (app_id,))
        self.assertEqual(self.cursor.fetchone()[0], 'Confirmed')
        self.cursor.execute('SELECT amount_due, payment_status FROM billing WHERE appointment_id = ?', (app_id,))
        bill = self.cursor.fetchone()
        self.assertEqual(bill[0], 150.0)
        self.assertEqual(bill[1], 'Unpaid')

    def test_double_booking_prevention(self):
        def check_and_book(pid, did, date, slot):
            self.cursor.execute('SELECT COUNT(*) FROM appointments WHERE doctor_id = ? AND appointment_date = ? AND time_slot = ?', (did, date, slot))
            if self.cursor.fetchone()[0] > 0:
                raise ValueError("Double booking!")
            self.cursor.execute('INSERT INTO appointments (patient_id, doctor_id, appointment_date, time_slot) VALUES (?, ?, ?, ?);', (pid, did, date, slot))
            self.conn.commit()
            
        check_and_book(1, 1, '2026-06-25', '10:00 AM')
        with self.assertRaises(ValueError):
            check_and_book(1, 1, '2026-06-25', '10:00 AM')

    def tearDown(self):
        self.conn.close()

# Run the tests in the Colab Cell
suite = unittest.TestLoader().loadTestsFromTestCase(TestCarePulse)
runner = unittest.TextTestRunner(verbosity=2)
print('--- Test Suite Report ---')
runner.run(suite)

## Phase 6 — Deployment and Monitoring
We simulate a 10-minute traffic load window, injecting an API load spike during minutes 4 to 6. When thresholds are breached, auto-scale mechanisms trigger automatically.

In [ ]:
print('--- CarePulse API Traffic Monitoring Simulation ---')
minutes = list(range(1, 11))

for m in minutes:
    # Base response times under normal load
    if 4 <= m <= 6:
        # Inject load spike
        response_time = round(np.random.normal(1250, 150), 2)
        alert_log = f" [ALERT] HTTP 504 Risk: Response time exceeded threshold (500ms)! Current: {response_time}ms"
        if m == 4:
            action_log = "\n   [AUTO-SCALE] Triggered horizontal scaling. Spinning up 2 API worker containers..."
        elif m == 5:
            action_log = "\n   [CACHING] Routing doctor catalog read queries to memory Redis cache..."
        else:
            action_log = ""
    else:
        response_time = round(np.random.normal(145, 10), 2)
        alert_log = ""
        action_log = ""
        
    print(f"Minute {m:02d}: Avg Response Time = {response_time:7.2f} ms{alert_log}{action_log}")

## Phase 7 — Maintenance and Prediction
We track a simulated 30-day timeline of server capacity metrics and train a linear regression model to project the exact day Database Load (%) hits the critical 90% threshold.

In [ ]:
# 1. Last 7 Days of Metrics Table
metrics_data = {
    'Day': [24, 25, 26, 27, 28, 29, 30],
    'Daily Users': [1150, 1195, 1240, 1290, 1335, 1380, 1420],
    'Error Rate (%)': [0.12, 0.15, 0.18, 0.14, 0.22, 0.25, 0.31],
    'Database Load (%)': [74.5, 76.2, 78.8, 80.1, 82.5, 83.9, 86.2],
    'Avg Response Time (ms)': [162, 168, 175, 170, 185, 192, 205]
}
df_metrics = pd.DataFrame(metrics_data)
print('--- Last 7 Days of Server Metrics ---')
display(df_metrics)

# 2. Regression Forecasting (30 days trend)
days = np.array(range(1, 31)).reshape(-1, 1)
# Simulate DB Load increasing linearly + some noise
np.random.seed(42)
db_load_30 = 35.0 + 1.8 * np.array(range(1, 31)) + np.random.normal(0, 1.2, 30)

reg = LinearRegression()
reg.fit(days, db_load_30)

slope = reg.coef_[0]
intercept = reg.intercept_

# Predict when CPU load reaches 90%
day_90 = int(np.ceil((90.0 - intercept) / slope))

print(f"\nLinear Regression Model:")
print(f"  Formula: DB Load (%) = {slope:.3f} * Day + {intercept:.3f}")
print(f"  CPU growth rate: {slope:.2f}% per day.")
print(f"  Projected day to hit 90% load: Day {day_90}")

# Plot
plt.figure(figsize=(10, 5))
plt.scatter(days, db_load_30, color='indigo', label='Historical CPU Load (%)')
future_days = np.array(range(1, day_90 + 5)).reshape(-1, 1)
predicted_load = reg.predict(future_days)
plt.plot(future_days, predicted_load, color='teal', linestyle='--', label='Regression Forecast')
plt.axhline(90, color='red', linestyle=':', label='90% Critical Threshold')
plt.axvline(day_90, color='red', linestyle='-.', label=f'Breach (Day {day_90})')
plt.title('CarePulse Database CPU Capacity Forecasting')
plt.xlabel('Operation Day')
plt.ylabel('Database CPU Load (%)')
plt.legend()
plt.tight_layout()
plt.savefig('outputs_p2_capacity.png', dpi=150)
plt.show()